# Trimming the chat history

In [19]:
from langchain_core.messages import SystemMessage, AIMessage, trim_messages, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from operator import itemgetter


In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [21]:
load_dotenv()

True

In [22]:
model = ChatOpenAI(model='gpt-4o')

In [23]:
trimmer = trim_messages(
    max_tokens = 50, 
    token_counter=model, 
    include_system=True, 
    allow_partial=False,
    start_on='human'
    )

In [24]:
messages = [
    HumanMessage(content='Hello'),
    AIMessage(content='Hi There!'),
    HumanMessage(content='Tell me a story about AI.'),
    AIMessage(content='Once upon a time... very long response...'),
    HumanMessage(content='Can you summarise it?'),
    AIMessage(content='Sure I can do it so as I was saying that there was a time...'),
]

In [25]:
trimmer.invoke(messages)

[HumanMessage(content='Can you summarise it?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sure I can do it so as I was saying that there was a time...', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [26]:
parser = StrOutputParser()

In [27]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Helpful AI Assistant and you have to remember the past conversation that we have spoken and on that basis you need to answer'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])

In [28]:
chain = RunnablePassthrough.assign(history= itemgetter("history") | trimmer) | prompt | model | parser

In [29]:
chain.invoke({'history': messages, 'input':'on which topic i asked for a story'})

"You asked for a story about a haunted house. I crafted a tale involving a mysterious old manor filled with unexplained occurrences, ghostly apparitions, and a curious protagonist determined to uncover the secrets behind its eerie reputation. If you'd like more details or a different story, feel free to ask!"